# 01 — Data Ingestion & Schema Validation

**Purpose**: Load the raw police violation CSV, validate the schema, cast dtypes, drop excluded columns, and deduplicate.

**Run this notebook cell by cell.** Each cell is independent and prints its own output.

**What you will see**:
- Schema validation results (pass/fail + warnings)
- Dtype cast confirmation
- Null summary for all retained columns
- Deduplication stats
- Final shape and split preview

**Files saved**:
- `data/processed/validation_report.json` — schema validation report
- `data/processed/load_metadata.json` — load stats, file hash, split preview

## Cell 1 — Environment setup
**What this cell does**: Adds the project root to `sys.path` so `src.*` imports work from the notebook.
**Expected output**: No errors. `Project root: ...GridLock R2` printed.

In [ ]:
import sys
from pathlib import Path

# Resolve project root (one level above notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'sys.path[0]: {sys.path[0]}')

## Cell 2 — Configure loguru
**What this cell does**: Sets up loguru to print readable log lines in the notebook output.
**Expected output**: `Loguru configured.`

In [ ]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='DEBUG',
    colorize=False,
)
print('Loguru configured.')

## Cell 3 — Schema Validation
**What this cell does**: Calls `validate_schema()` directly on the raw CSV (before any dtype casting).

**Expected output**: `─── Schema Validation PASSED (2 warning(s)) ─────`

Expected warnings (normal):
- 5 unparseable `created_datetime` rows (will be dropped in load.py)
- Leakage columns present (will be dropped in load.py)

If this cell raises `ValueError`, stop and fix before continuing.

In [ ]:
import pandas as pd
from src.data.validate import validate_schema, load_eval_config, save_validation_report

CSV_PATH      = PROJECT_ROOT / 'data' / 'raw' / 'jan to may police violation_anonymized791b166.csv'
EVAL_CFG_PATH = PROJECT_ROOT / 'configs' / 'eval.yaml'

print(f'Loading raw CSV from: {CSV_PATH}')
df_raw   = pd.read_csv(CSV_PATH, dtype=str, low_memory=False)
print(f'Raw shape: {df_raw.shape}')

eval_cfg = load_eval_config(EVAL_CFG_PATH)
report   = validate_schema(df_raw, eval_cfg, strict=True)
save_validation_report(report, PROJECT_ROOT / 'data' / 'processed' / 'validation_report.json')

print(f'\nValidation passed: {report["passed"]}')
print(f'Warnings: {len(report["warnings"])}')
print(f'Errors:   {len(report["errors"])}')

## Cell 4 — Full ingest with load_raw()
**What this cell does**: Calls `load_raw()` — validate + cast dtypes + drop columns + deduplicate.

**Expected output**:
- Progress bars for each step
- `298,450 → 268,281 rows (30,169 exact duplicates removed)`
- `Cols retained: 12`

In [ ]:
from src.data.load import load_raw, save_load_metadata

df, metadata = load_raw(
    csv_path=CSV_PATH,
    eval_config_path=EVAL_CFG_PATH,
    save_report=True,
    report_path=PROJECT_ROOT / 'data' / 'processed' / 'validation_report.json',
)
save_load_metadata(metadata, PROJECT_ROOT / 'data' / 'processed' / 'load_metadata.json')

print(f'\nFinal DataFrame shape: {df.shape}')
print(f'Columns retained: {list(df.columns)}')

## Cell 5 — Dtype sanity check
**Expected output**:
- `created_datetime` = `datetime64[ns, UTC]`
- `latitude`, `longitude` = `float64`
- `data_sent_to_scita` = `int8`
- `center_code` ~3.77% nulls (expected — imputed in features.py)

In [ ]:
print('=== dtypes ===')
print(df.dtypes)

print('\n=== head(3) ===')
print(df.head(3).to_string())

print('\n=== null counts (retained columns) ===')
null_counts = df.isna().sum()
print(null_counts[null_counts > 0] if null_counts.any() else 'No nulls in retained columns ✓')

print('\n=== violation_type sample ===')
print(df['violation_type'].head(5).tolist())

## Cell 6 — Date range + split preview
**Expected output**:
```
Date range: 2023-11-09 → 2024-04-08
Train rows (≤ 2024-02-29): ~226k
Test rows  (2024-03-01 – 2024-04-08): ~70k
```

In [ ]:
import yaml
with open(EVAL_CFG_PATH) as f:
    eval_local = yaml.safe_load(f)
split      = eval_local['split']
train_end  = pd.Timestamp(split['train_end'],  tz='UTC')
test_start = pd.Timestamp(split['test_start'], tz='UTC')
test_end   = pd.Timestamp(split['test_end'],   tz='UTC')

n_train = (df['created_datetime'] <= train_end).sum()
n_test  = ((df['created_datetime'] >= test_start) & (df['created_datetime'] <= test_end)).sum()

print(f"Date range  : {df['created_datetime'].min().date()} → {df['created_datetime'].max().date()}")
print(f"Train rows  : {n_train:,}  (≤ {train_end.date()})")
print(f"Test rows   : {n_test:,}  ({test_start.date()} – {test_end.date()})")
print(f"Dedup removed: {metadata['rows_dropped_dedup']:,} rows")

## Cell 7 — Confirm null columns dropped
**Expected output**: All three show `NOT PRESENT (correctly dropped) ✓`

In [ ]:
for col in ['description', 'closed_datetime', 'action_taken_timestamp']:
    status = '⚠  STILL PRESENT' if col in df.columns else '✓  NOT PRESENT (correctly dropped)'
    print(f'  {status}: {col}')

## Cell 8 — Confirm leakage columns dropped
**Expected output**: All show `NOT PRESENT (correctly excluded) ✓`

In [ ]:
for col in ['data_sent_to_scita_timestamp', 'modified_datetime', 'validation_status',
            'validation_timestamp', 'updated_vehicle_number', 'updated_vehicle_type']:
    status = '⚠  STILL PRESENT — leakage risk!' if col in df.columns else '✓  NOT PRESENT (correctly excluded)'
    print(f'  {status}: {col}')

## Summary

**What was done**:
- Raw CSV loaded and schema-validated (8 checks)
- Dtypes cast: `created_datetime` → UTC datetime64, `latitude`/`longitude` → float64, `data_sent_to_scita` → int8
- 12 excluded columns dropped (null + leakage + identifier)
- Deduplicated by minute-level rule (same-second events at different lat/lon kept)

**What was saved**:
- `data/processed/validation_report.json`
- `data/processed/load_metadata.json`

**Next step**: Run `notebooks/01b_features.ipynb` — feature engineering (Phase A)

In [ ]:
print('=== Session Summary ===')
print(f'  Source file     : {metadata["source_file"]}')
print(f'  SHA-256 (16)    : {metadata["file_hash_sha256"][:16]}...')
print(f'  Rows raw        : {metadata["rows_raw"]:,}')
print(f'  Rows after dedup: {metadata["rows_final"]:,}')
print(f'  Cols retained   : {metadata["cols_final"]}')
print(f'  Validation      : {"PASSED" if metadata["validation_passed"] else "FAILED"}')
print(f'  Saved           : data/processed/validation_report.json')
print(f'  Saved           : data/processed/load_metadata.json')
print(f'  Next notebook   : notebooks/01b_features.ipynb')